# Setup

In [ ]:
# Install QP solver for GEM
!pip install quadprog

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 534.6/534.6 kB 21.8 MB/s eta 0:00:00


In [ ]:
# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import quadprog
import random

# Orthogonal Gradient Descent (OGD)

OGD mitigates catastrophic forgetting by projecting the gradients of the current task onto a subspace that is orthogonal to the gradients of the neural network's predictions on previous tasks.

This implementation uses the "OGD-GTL" (Ground Truth Logit) variant, which stores the gradients with respect to the ground truth logit to reduce memory requirements. When training on a new task, it modifies the original gradient $g$ to a new gradient $\tilde{g}$ that is orthogonal to the stored basis $S$, calculated as $\tilde{g} = g - \sum_{i=1}^{n_A} \text{proj}_{v_i}(g)$.

In [ ]:
class OGD:
    def __init__(self, model, max_basis_size=200):
        """
        Orthogonal Gradient Descent for Continual Learning.
        """
        self.model = model
        self.basis = [] # Stores the orthogonalized gradient basis vectors (S)
        self.max_basis_size = max_basis_size
        self.device = next(model.parameters()).device

    def _get_flattened_gradients(self):
        """Helper to extract and flatten all current gradients of the model."""
        grads = []
        for param in self.model.parameters():
            if param.grad is not None:
                grads.append(param.grad.view(-1))
        return torch.cat(grads)

    def _set_flattened_gradients(self, flat_grads):
        """Helper to assign a flattened gradient vector back to the model."""
        offset = 0
        for param in self.model.parameters():
            if param.grad is not None:
                numel = param.numel()
                param.grad.copy_(flat_grads[offset:offset + numel].view(param.shape))
                offset += numel

    def project_gradients(self):
        """
        Projects the current model gradients orthogonal to the stored basis.
        Call this after loss.backward() but before optimizer.step().
        """
        if not self.basis:
            return # No projection needed for the first task

        g = self._get_flattened_gradients()

        # g_tilde = g - sum(proj_v(g))
        for v in self.basis:
            # proj_v(g) = (<g, v> / <v, v>) * v
            # Since basis vectors are normalized, <v, v> = 1
            dot_product = torch.dot(g, v)
            g = g - dot_product * v

        self._set_flattened_gradients(g)

    def update_basis(self, task_dataloader):
        """
        Computes the gradient of the model predictions (ground-truth logit) on a subset
        of the task data and adds them to the orthogonal basis using Gram-Schmidt.
        Call this at the END of a task.
        """
        self.model.eval()
        added_vectors = 0

        for x, y in task_dataloader:
            x, y = x.to(self.device), y.to(self.device)

            # Iterate sample by sample to get individual gradients
            for i in range(x.size(0)):
                if len(self.basis) >= self.max_basis_size:
                    return

                self.model.zero_grad()
                logits = self.model(x[i:i+1])

                # OGD-GTL: Compute gradient w.r.t. ground truth logit
                ground_truth_logit = logits[0, y[i]]
                ground_truth_logit.backward()

                # Extract gradient
                u = self._get_flattened_gradients()

                # Gram-Schmidt Orthogonalization against existing basis
                for v in self.basis:
                    dot_product = torch.dot(u, v)
                    u = u - dot_product * v

                # Normalize and add to basis if not a zero vector
                norm = torch.norm(u)
                if norm > 1e-5:
                    u = u / norm
                    self.basis.append(u.detach().clone())
                    added_vectors += 1

# Gradient Episodic Memory (GEM)

GEM solves a dual quadratic program (QP) on the variables representing past tasks to ensure that the current gradient update does not increase the loss on examples stored in the episodic memory.

If one or more inequality constraints are violated (meaning the gradient would increase the loss on past tasks), GEM projects the gradient to the closest vector $\tilde{g}$ solving the QP: minimize $\frac{1}{2} v^\top G G^\top v + g^\top G^\top v$ subject to $v \ge 0$.

In [ ]:
class GEM:
    def __init__(self, model, memory_budget=5120, gamma=0.1):
        """
        Gradient Episodic Memory for Continual Learning.
        """
        self.model = model
        self.memory_budget = memory_budget
        self.gamma = gamma # Small constant biased to favor beneficial backwards transfer
        self.episodic_memory = {} # Maps task_id -> list of (x, y) tuples
        self.device = next(model.parameters()).device

    def _get_flattened_gradients(self):
        grads = []
        for param in self.model.parameters():
            if param.grad is not None:
                grads.append(param.grad.view(-1))
        return torch.cat(grads)

    def _set_flattened_gradients(self, flat_grads):
        offset = 0
        for param in self.model.parameters():
            if param.grad is not None:
                numel = param.numel()
                param.grad.copy_(flat_grads[offset:offset + numel].view(param.shape))
                offset += numel

    def project_gradients(self, current_task_id, current_loss_fn):
        """
        Checks gradients against episodic memory. If constraints are violated,
        solves the QP to project the gradients.
        Call this after loss.backward() but before optimizer.step().
        """
        if not self.episodic_memory:
            return # No past tasks to conflict with

        # 1. Store the gradient for the current task (g)
        g = self._get_flattened_gradients().detach().clone()

        past_task_grads = []

        # 2. Compute gradients on episodic memories of past tasks (g_k)
        for task_id, memory in self.episodic_memory.items():
            if task_id >= current_task_id:
                continue

            self.model.zero_grad()
            # In practice, memory should be pre-batched. Assuming 'memory' is a tuple (X, Y)
            x_mem, y_mem = memory[0].to(self.device), memory[1].to(self.device)

            logits = self.model(x_mem)
            loss_k = current_loss_fn(logits, y_mem)
            loss_k.backward()

            past_task_grads.append(self._get_flattened_gradients().detach())

        if not past_task_grads:
            self._set_flattened_gradients(g)
            return

        # Stack past gradients into a matrix G
        G = torch.stack(past_task_grads) # Shape: (num_past_tasks, num_parameters)

        # 3. Check for constraint violations: <g, g_k> >= 0
        dot_products = torch.mv(G, g)

        if (dot_products >= 0).all():
            self._set_flattened_gradients(g) # No violation
            return

        # 4. If violation, solve Dual QP
        # min 0.5 * v^T (G * G^T) v + g^T G^T v subject to v >= 0
        G_np = G.cpu().numpy()
        g_np = g.cpu().numpy()

        # P = G * G^T
        P = np.dot(G_np, G_np.T)
        # Add small constant to diagonal for numerical stability and biased backward transfer
        P = P + self.gamma * np.eye(P.shape[0])

        # q = G * g
        q = np.dot(G_np, g_np)

        # Inequality constraints: v >= 0  => A^T v >= b
        A = np.eye(P.shape[0])
        b = np.zeros(P.shape[0])

        try:
            # quadprog solves: min 0.5 * x^T P x - q^T x s.t. A^T x >= b
            # We negate q to match quadprog's formulation
            v_star = quadprog.solve_qp(P, -q, A, b)[0]
        except ValueError:
            # Fallback if QP is unsolvable (usually numerical issues)
            v_star = np.zeros(P.shape[0])

        # 5. Recover projected gradient: g_tilde = G^T v* + g
        v_star = torch.tensor(v_star, dtype=torch.float32, device=self.device)
        g_tilde = torch.mv(G.t(), v_star) + g

        self._set_flattened_gradients(g_tilde)

    def update_memory(self, task_id, x_data, y_data):
        """
        Populate the episodic memory for a task.
        Call this at the END of a task, or handle memory building organically.
        """
        # Truncate to match memory budget across T tasks
        # (Assuming your team handles the matched sampling procedure)
        self.episodic_memory[task_id] = (x_data, y_data)

# Gradient Surgery (PCGrad)

Since PCGrad was originally designed for multi-task learning, adapting it to continual learning means treating the current-task objective and the replay objective as two separate, potentially conflicting signals.

The core logic of PCGrad is that, if two gradients conflict (meaning their dot product is negative), then project one gradient onto the normal of the other. Mathematically, if $g_i \cdot g_j < 0$, the projected gradient becomes:$$g_i = g_i - \frac{g_i \cdot g_j}{\|g_j\|^2} g_j$$

For our adaptation, we compute the gradient of the current batch and the gradient of the replay batch. We then shuffle their order to prevent optimization bias and apply the projection above if they conflict. Finally, we sum the projected gradients to update the model.

In [ ]:
class PCGrad:
    def __init__(self, model):
        """
        PCGrad (Gradient Surgery) adapted for Continual Learning.
        """
        self.model = model
        self.episodic_memory = [] # Stores the replay buffer batch
        self.device = next(model.parameters()).device

    def _get_flattened_gradients(self):
        """Helper to extract and flatten all current gradients of the model."""
        grads = []
        for param in self.model.parameters():
            if param.grad is not None:
                grads.append(param.grad.view(-1))
        return torch.cat(grads)

    def _set_flattened_gradients(self, flat_grads):
        """Helper to assign a flattened gradient vector back to the model."""
        offset = 0
        for param in self.model.parameters():
            if param.grad is not None:
                numel = param.numel()
                param.grad.copy_(flat_grads[offset:offset + numel].view(param.shape))
                offset += numel

    def project_gradients(self, current_loss_fn):
        """
        Treats the current task batch and the episodic memory batch as two
        separate tasks. Applies gradient surgery if their gradients conflict.
        Call this after loss.backward() but before optimizer.step().
        """
        if not self.episodic_memory:
            return # No past tasks to conflict with

        # 1. Store the gradient for the current task (g_current)
        g_current = self._get_flattened_gradients().detach().clone()

        # 2. Compute the gradient on the replay memory (g_replay)
        self.model.zero_grad()

        # In a real pipeline, the memory should be sampled/batched by your dataloader.
        # Here we assume episodic_memory contains a unified batch (X, Y) of past tasks.
        x_mem, y_mem = self.episodic_memory[0].to(self.device), self.episodic_memory[1].to(self.device)

        logits = self.model(x_mem)
        loss_replay = current_loss_fn(logits, y_mem)
        loss_replay.backward()

        g_replay = self._get_flattened_gradients().detach().clone()

        # 3. PCGrad logic for T=2 (Current Task vs. Replay Memory)
        grads = [g_current, g_replay]
        projected_grads = []

        # Randomize order to avoid optimization bias toward either task
        indices = [0, 1]
        random.shuffle(indices)

        for i in indices:
            g_i = grads[i].clone()
            g_j = grads[1 - i]

            # Check for conflict
            dot_product = torch.dot(g_i, g_j)
            if dot_product < 0:
                # Conflict detected: project g_i onto the normal of g_j
                norm_sq = torch.dot(g_j, g_j)
                if norm_sq > 1e-8: # Prevent division by zero
                    g_i = g_i - (dot_product / norm_sq) * g_j

            projected_grads.append(g_i)

        # Sum the projected gradients (g_final = g_cur^proj + g_rep^proj)
        g_final = projected_grads[0] + projected_grads[1]

        # 4. Set the final surgical gradient back to the model
        self._set_flattened_gradients(g_final)

    def update_memory(self, x_data, y_data):
        """
        Update the replay buffer.
        For this PCGrad CL adaptation, your team's data processing module
        should ideally pass a balanced subsample of all previous tasks here.
        """
        self.episodic_memory = (x_data, y_data)